In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
PARQ_PATH = Path('../sample_dataset/processed_data/parquets/20220730-0002.parquet')
DC_OFFSET = 1.9  # Volts

In [ ]:
# Load the parquet file into a DataFrame
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype('int16') # parquet stores column names as strings; we want integers

num_initial_signals = df.shape[0]
print(f"\x1b[1;36m{num_initial_signals}\x1b[0m signals loaded.")

In [ ]:
# Drop any signals that have missing or infinite values
df = df.replace([np.inf, -np.inf], np.nan).dropna(how='any')
num_missing_signals = num_initial_signals - df.shape[0]
print(f"Removed \x1b[1;31m{num_missing_signals}\x1b[0m signals with missing or infinite values.")

In [ ]:
# Subtract DC offset
df = df - DC_OFFSET
# Invert signals
df = -1 * df

In [ ]:
# Offset dataframe upwards by global minimum
# TODO: check if this is necessary
df = df + abs(min(df.min()))

In [ ]:
# Plot signals
fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df.T
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

plt.show()

In [ ]:
filt = (df[0] < 0.15)
bad_filt = (df[0] >= 0.15)
df_dirty = df[bad_filt]

fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df_dirty.T
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

plt.show()

In [ ]:
import scipy.signal as sps

# IMPORTANT ONES TO CHECK IN `df_dirty`:
# 51

start = 51
stop = start + 2
win_len = 10
po = 3
der = 1


clean_idx = 10
dirty_idx = 51
df_smooth = sps.savgol_filter(
    df_dirty.iloc[dirty_idx,:], 
    window_length=win_len, 
    polyorder=po, 
    deriv=der
)

df_smooth_og = sps.savgol_filter(
    df.iloc[clean_idx,:], 
    window_length=win_len, 
    polyorder=po, 
    deriv=der
)

fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df.iloc[clean_idx,:].T, 'b-', label='Standard Wave'
)

ax.plot(
    df_smooth_og.T, 'b--', label='Derv of Standard'
)


ax.plot(
    df_dirty.iloc[dirty_idx,:].T, 'k-', label='Bad Wave'
)

ax.plot(
    df_smooth, 'k--', label='1st Deriv of Bad after smoothing'
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

plt.legend()
plt.show()

# Gaussian Fitting


In [ ]:
import scipy as sp
import scipy.stats as stats
from scipy import optimize

In [ ]:
data = df[0:5].T
results = df.apply(stats.norm.fit, axis=1)

fig, ax = plt.subplots(figsize=(15,8))
    
scale = np.linspace(0, data.shape[0])
for res in results: 
    fit = stats.norm.pdf(scale, res[0], res[1])
    ax.plot(scale, fit)
    
ax.grid(visible=True)
ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

data = df.iloc[2,:]  # Try playing with index here! Some work better than others...(eg: index 1 vs 2)
data = sps.savgol_filter(data, window_length=25, polyorder=3)
x = np.linspace(0, data.shape[0], data.shape[0])

def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) / 4 / stddev)**2)

popt, _ = optimize.curve_fit(gaussian, x, data)

plt.plot(x, data)
plt.plot(x, gaussian(x, *popt))

# Smoothing
Using Savitzky-Golay Filter

In [ ]:
window_length = 10
polyorder = 3

df_smooth = df.T.apply(
    sps.savgol_filter, args=(window_length, polyorder)
)

df_dirty_smooth = df_smooth[df[bad_filt].index]

## Plot Smoothed Data

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df_smooth
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

plt.show()

In [ ]:
def find_peaks(series, prominence=0.1):
    peaks, amplitudes = sps.find_peaks(series, prominence=prominence)
    return peaks, amplitudes['prominences']

packed_vals = df_smooth.apply(find_peaks)
peaks, amplitudes = packed_vals.iloc[0,:], packed_vals.iloc[1,:]
to_drop = [*peaks[peaks.apply( len ) == 0 ].index]
df_smooth_filtered = df_smooth.drop(to_drop, axis=1)
amplitudes = amplitudes.drop(to_drop)
num_missing_signals = num_initial_signals - df_smooth_filtered.shape[1]
print(f"Removed \x1b[1;31m{num_missing_signals}\x1b[0m signals assumed to be bad.")

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df_smooth_filtered
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

plt.show()

In [ ]:
offset = 5
tail_indices = peaks + offset

def sum_tail(series, tail_indices = tail_indices):
    tail_idx = tail_indices[series.name]
    return series.iloc[tail_idx[0]:-1].sum()

sums = df_smooth_filtered.sum()
tail_sum = df_smooth_filtered.apply(sum_tail)

In [ ]:
quotients = tail_sum / sums
quotients

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

ax.plot(sums, tail_sum, 'o', markersize=1)
plt.xlabel('Total Integral')
plt.ylabel('Tail Integral')

# Time Analysis


In [ ]:
%%timeit
window_length = 10
polyorder = 3
offset = 5

df_smooth = df.T.apply(
    sps.savgol_filter, args=(window_length, polyorder)
)

def find_peaks(series, prominence=0.1):
    peaks, _ = sps.find_peaks(series, prominence=prominence)
    return peaks

peaks = df_smooth.apply(find_peaks)
to_drop = peaks[peaks.apply( len ) == 0 ].index
df_smooth_filtered = df_smooth.drop([*peaks[peaks.apply( len ) == 0 ].index], axis=1)

tail_indices = peaks + offset

def sum_tail(series, tail_indices = tail_indices):
    tail_idx = tail_indices[series.name]
    return series.iloc[tail_idx[0]:-1].sum()

sums = df_smooth_filtered.sum()
tail_sum = df_smooth_filtered.apply(sum_tail)

In [ ]:
df